# Corr_labpm: exact Gillespie ensembles and Chebyshev-shell NEEP

Four orientations (up, right, down, left), periodic exclusion, and a non-reciprocal forward hopping rate:
\[
w_{i,+}=v_+^0\exp\!\left[\sum_{k=2}^{R}\beta_k n(\mathbf r_i+k\mathbf e_i)\right].
\]
The destination must be empty. Every occupied site on the forward ray contributes, without a nearest-particle or visibility test. Increasing \(\beta_k\) gives more distant particles a larger effect. Backward and lateral rates are constant; each of the two quarter-turns has rate \(\gamma\).

The simulator uses **exact event-driven Gillespie dynamics**, with independent replicas parallelized on CPU workers or in CUDA batches. Each replica retains its own Gillespie clock and executes individual events; batching does not introduce a common integration step. This notebook trains on multiple independent trajectories and keeps validation/test replicas separate. It uses no entropy-production labels in training.

Set `LABPM_BACKEND` to `auto` (default), `cpu`, or `cuda` for simulation. CUDA simulation availability is checked through Numba independently of the PyTorch training device. `CUDA_BATCH_SIZE` controls the number of replicas in a GPU batch. CPU workers are selected automatically; set `LABPM_WORKERS` to override their number.

Run all cells in order. Set the environment variable `LABPM_SMOKE=1` for a small end-to-end execution check; those settings do not establish learning or stationarity.

In [ ]:
import os
import sys
import json
import math
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import asdict
from types import SimpleNamespace

def find_project_root():
    for directory in (Path.cwd(), *Path.cwd().parents):
        for candidate in (directory, directory / "CNEEP_v2"):
            if (candidate / "models" / "NEEP_ShellForce_2D.py").is_file() and (candidate / "data").is_dir():
                return candidate.resolve()
    raise FileNotFoundError("Run this notebook from CNEEP_v2, its notebooks directory, or the repository workspace.")

ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from tqdm.auto import trange
from data.labp_m import LABPMConfig, simulate_ensemble, encode_observations, simulation_backend
from models.NEEP_ShellForce_2D import MultiScaleShellForceCNEEP2D

SMOKE = os.environ.get("LABPM_SMOKE", "0") == "1"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
torch.manual_seed(2026)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(2026)
print("Project:", ROOT)
print("Training device:", DEVICE, "| smoke:", SMOKE)

## 1. Physics, ensemble size, and observation times

All rates have units of inverse physical time. Lattice spacing and Boltzmann's constant are one. The actual particle count is set by the simulator from the density. The nonzero backward rate keeps every allowed hop's reverse rate positive.

These defaults are a pilot experiment. Increase burn time and observation duration after inspecting relaxation; increase training duration after inspecting held-out objectives. Set `SAMPLE_DT` to the **fixed physical lag of every training pair** (`sampling frequency = 1 / SAMPLE_DT`). `OBSERVATION_TIME` sets the post-burn physical duration; the number of frames is derived automatically, so changing the sampling frequency does not change that duration. All train/validation/test pairs use this same lag. Gillespie event waiting times remain random internally; they are never used as the training lag. Four direction occupancy channels retain the particle orientation without putting the precomputed interaction into a local channel.

In [ ]:
config = LABPMConfig(
    lattice_size=10 if SMOKE else 24,
    density=0.35,
    forward_rate=2.0,
    backward_rate=0.2,
    lateral_rate=0.2,
    rotation_rate=0.1,
    shell_weights=(0.15, 0.30, 0.45),  # beta_2, beta_3, beta_4
)
TRAIN_M, VAL_M, TEST_M = (2, 2, 2) if SMOKE else (16, 4, 4)
SAMPLE_DT = 0.02 if SMOKE else 0.005  # fixed physical training lag
OBSERVATION_TIME = 0.32 if SMOKE else 5.0  # fixed post-burn duration
if not (math.isfinite(SAMPLE_DT) and SAMPLE_DT > 0
        and math.isfinite(OBSERVATION_TIME) and OBSERVATION_TIME > 0):
    raise ValueError("SAMPLE_DT and OBSERVATION_TIME must be finite and positive.")
n_intervals = int(round(OBSERVATION_TIME / SAMPLE_DT))
if n_intervals < 2 or not math.isclose(n_intervals * SAMPLE_DT, OBSERVATION_TIME, rel_tol=1e-10, abs_tol=1e-12):
    raise ValueError("OBSERVATION_TIME must be an integer multiple of SAMPLE_DT with at least two intervals.")
N_FRAMES = n_intervals + 1
SAMPLING_FREQUENCY = 1.0 / SAMPLE_DT
BURN_TIME = 0.1 if SMOKE else 100.0
SIMULATION_BACKEND = os.environ.get("LABPM_BACKEND", "auto").strip().lower()
RESOLVED_SIMULATION_BACKEND = simulation_backend(SIMULATION_BACKEND)
CUDA_BATCH_SIZE = 64
WORKERS = int(os.environ["LABPM_WORKERS"]) if "LABPM_WORKERS" in os.environ else None  # None uses available CPUs
SPLIT_SEEDS = {"train": 12031, "validation": 23041, "test": 34057}
CACHE_DATA = not SMOKE
FORCE_RESIMULATE = False

# Each shell branch only sees that exclusive Chebyshev shell.
R = len(config.shell_weights) + 1
AREA = config.lattice_size**2
MODEL_MAX_DISTANCE = min(R + 1, (config.lattice_size - 1) // 2)
opt = SimpleNamespace(
    n_components=4, n_channel=8 if SMOKE else 32, n_hidden=2,
    max_distance=MODEL_MAX_DISTANCE, include_k0=True,
    positional=False, beta=1.0, shell_center_mode="relative_only",
    shell_relative_mode="learned_absolute", shell_force_bias=True,
    shell_weight_normalization="none", shell_force_activation="elu",
)
N_ITER = 3 if SMOKE else 1500
BATCH_SIZE = 8 if SMOKE else 64
EVAL_BATCH_SIZE = 16 if SMOKE else 128
RECORD_EVERY = 1 if SMOKE else 50
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-5
ALPHA = -0.5
CLIP_NORM = 10.0
TRAIN_SEED = 2026

if min(TRAIN_M, VAL_M, TEST_M) < 2:
    raise ValueError("Use at least two independent trajectories in each split; TRAIN_M=1 is deliberately disallowed.")
assert len(set(SPLIT_SEEDS.values())) == 3
assert MODEL_MAX_DISTANCE < config.lattice_size / 2
RESULT_BASE = Path(os.environ.get("LABPM_OUTPUT_DIR", str(ROOT / "results"))).expanduser()
RESULT_DIR = RESULT_BASE / ("Corr_labpm_smoke" if SMOKE else "Corr_labpm") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
RESULT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = ROOT / "data" / "labp_m" / "cache"
print(json.dumps(asdict(config), indent=2))
print(f"Simulation backend: {RESOLVED_SIMULATION_BACKEND} (requested {SIMULATION_BACKEND}); training device: {DEVICE}")
print(f"CPU workers={WORKERS if WORKERS is not None else 'auto'}; CUDA replica batch size={CUDA_BATCH_SIZE}")
print(f"Independent replicas: train={TRAIN_M}, validation={VAL_M}, test={TEST_M}")
print(f"Saved duration={(N_FRAMES - 1) * SAMPLE_DT:g}, burn={BURN_TIME:g}, Chebyshev k=0..{MODEL_MAX_DISTANCE}")
print(f"Every training pair: delta_t={SAMPLE_DT:g}; sampling frequency={SAMPLING_FREQUENCY:g} / time")
print("Results:", RESULT_DIR)

## 2. Simulate independent replicas or load matching caches

Cache keys include all physics, seeds, observation settings, the resolved simulation backend, and hashes of the simulator's Python sources including the CUDA implementation. Replica boundaries remain explicit as `[ensemble, time, ...]`. Re-running with the same backend and seed produces the same replicas independently of CPU worker count. CPU and CUDA use different random-number generators, so their individual trajectories need not match.

The event-level reference is
\[
\Delta s_{\mathrm{med}}=\log[w(X\to X')/w(X'\to X)].
\]
The reverse rate is evaluated in the post-event state. Orientations are even under time reversal; symmetric rotations contribute zero. The result is the medium path entropy of this specified Markov model. It is not a claim about unmodeled physical dissipation.

In [ ]:
def source_hash(directory):
    digest = hashlib.sha256()
    for path in sorted(directory.glob("*.py")):
        digest.update(path.name.encode())
        digest.update(path.read_bytes())
    return digest.hexdigest()

SIMULATOR_HASH = source_hash(ROOT / "data" / "labp_m")
cuda_source_path = ROOT / "data" / "labp_m" / "_gillespie_cuda.py"
CUDA_SOURCE_HASH = hashlib.sha256(cuda_source_path.read_bytes()).hexdigest() if cuda_source_path.is_file() else None
ARRAY_FIELDS = (
    "states", "times", "medium_ep", "medium_ep_maps",
    "shell_ep", "hop_counts", "rotation_counts", "seeds",
)
cache_ids = {}

def simulate_split(name, count):
    request = {
        "schema": 2, "config": asdict(config),
        "n_trajectories": count, "n_frames": N_FRAMES,
        "sample_dt": SAMPLE_DT, "burn_time": BURN_TIME,
        "seed": SPLIT_SEEDS[name], "simulator_sha256": SIMULATOR_HASH,
        "backend": RESOLVED_SIMULATION_BACKEND, "cuda_batch_size": CUDA_BATCH_SIZE,
        "cuda_source_sha256": CUDA_SOURCE_HASH,
    }
    metadata = json.dumps(request, sort_keys=True, separators=(",", ":"))
    cache_id = hashlib.sha256(metadata.encode()).hexdigest()[:20]
    cache_ids[name] = cache_id
    path = CACHE_DIR / f"{name}_{cache_id}.npz"
    if CACHE_DATA and path.is_file() and not FORCE_RESIMULATE:
        with np.load(path, allow_pickle=False) as archive:
            if str(archive["metadata"].item()) != metadata:
                raise ValueError(f"Cache metadata mismatch: {path}")
            result = SimpleNamespace(**{field: archive[field].copy() for field in ARRAY_FIELDS})
        print(f"{name}: loaded {count} independent replicas from {path.name}")
    else:
        result = simulate_ensemble(
            config, n_trajectories=count, n_frames=N_FRAMES,
            sample_dt=SAMPLE_DT, burn_time=BURN_TIME,
            seed=SPLIT_SEEDS[name], workers=WORKERS, progress=not SMOKE,
            backend=RESOLVED_SIMULATION_BACKEND, batch_size=CUDA_BATCH_SIZE,
        )
        if CACHE_DATA:
            CACHE_DIR.mkdir(parents=True, exist_ok=True)
            # Atomic replacement avoids leaving a valid-looking partial cache.
            temporary = path.with_suffix(".partial.npz")
            np.savez_compressed(temporary, metadata=np.asarray(metadata),
                                **{field: np.asarray(getattr(result, field)) for field in ARRAY_FIELDS})
            temporary.replace(path)
    assert result.states.shape == (count, N_FRAMES, config.lattice_size, config.lattice_size)
    assert result.medium_ep.shape == (count, N_FRAMES - 1)
    np.testing.assert_allclose(result.times, np.arange(N_FRAMES) * SAMPLE_DT, atol=1e-10)
    np.testing.assert_allclose(np.diff(result.times), SAMPLE_DT, rtol=1e-10, atol=1e-12)
    np.testing.assert_allclose(result.medium_ep_maps.sum(axis=(-2, -1)), result.medium_ep, atol=1e-9)
    np.testing.assert_allclose(result.shell_ep.sum(axis=-1), result.medium_ep, atol=1e-9)
    assert np.isfinite(result.medium_ep).all()
    return result

results = {name: simulate_split(name, count) for name, count in
           (("train", TRAIN_M), ("validation", VAL_M), ("test", TEST_M))}
all_seeds = np.concatenate([np.asarray(result.seeds) for result in results.values()])
assert len(np.unique(all_seeds)) == len(all_seeds), "Replica seeds overlap."
videos = {name: encode_observations(result.states).cpu() for name, result in results.items()}
train_video, val_video, test_video = (videos[name] for name in ("train", "validation", "test"))
for name, video in videos.items():
    assert video.dtype == torch.float32 and video.shape[2] == 4
    assert torch.all(video.sum(dim=2) <= 1)
    print(name, tuple(video.shape), "| mean hops/frame:",
          float(results[name].hop_counts.mean()), "| mean rotations/frame:",
          float(results[name].rotation_counts.mean()))

## 3. Snapshots and relaxation checks

The same trajectory is shown at three saved times. Adjacent-site occupancy correlation and medium entropy rates in the first/second halves are simple drift checks, not a proof of stationarity. Compare longer burn times and larger systems before interpreting an apparent cluster as MIPS.

In [ ]:
direction_cmap = ListedColormap(["white", "#377eb8", "#e41a1c", "#4daf4a", "#984ea3"])
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5), constrained_layout=True)
for ax, frame in zip(axes, (0, N_FRAMES // 2, N_FRAMES - 1)):
    image = ax.imshow(results["train"].states[0, frame] + 1, cmap=direction_cmap,
                      vmin=-0.5, vmax=4.5, interpolation="nearest")
    ax.set_title(f"Train replica 0, t={results['train'].times[frame]:.2f}")
    ax.set_xticks([]); ax.set_yticks([])
bar = fig.colorbar(image, ax=axes, ticks=range(5), shrink=0.85)
bar.ax.set_yticklabels(["empty", "up", "right", "down", "left"])
fig.savefig(RESULT_DIR / "direction_snapshots.png", dpi=150)
plt.show()

def mean_sem(values):
    values = np.asarray(values, dtype=np.float64)
    return float(values.mean()), float(values.std(ddof=1) / np.sqrt(len(values)))

def nearest_correlation(states):
    occupied = (states >= 0).astype(np.float64)
    pair = 0.5 * (occupied * np.roll(occupied, 1, axis=-1)
                  + occupied * np.roll(occupied, 1, axis=-2)).mean(axis=(-2, -1))
    rho = occupied.mean(axis=(-2, -1))
    return pair - rho**2

drift_diagnostics = {}
for name, result in results.items():
    mid = (N_FRAMES - 1) // 2
    first = result.medium_ep[:, :mid].mean(axis=1) / SAMPLE_DT
    second = result.medium_ep[:, mid:].mean(axis=1) / SAMPLE_DT
    corr = nearest_correlation(result.states)
    drift_diagnostics[name] = {
        "medium_rate_first_half": mean_sem(first),
        "medium_rate_second_half": mean_sem(second),
        "nearest_covariance_first_half": float(corr[:, :N_FRAMES // 2].mean()),
        "nearest_covariance_second_half": float(corr[:, N_FRAMES // 2:].mean()),
    }
print(json.dumps(drift_diagnostics, indent=2))

## 4. Ensemble-aware pair sampling and unsupervised training

A training sample selects a replica and an adjacent time pair. Validation evaluates every pair of its own replicas; no transition is formed across replica boundaries.

The model API returns a **spatial mean**, so this notebook multiplies branch outputs by the lattice area to obtain extensive branch entropy increments. Their sum is the dimensionless total score used in alpha-NEEP. The model's raw maps are local entropy increments: summing them over sites and branches gives exactly that total score. This convention keeps local learned forces independent of the lattice area. No time factor enters the objective; rates are reported afterward by dividing increments by physical time.

The final contraction uses increments of the four orientation occupancy channels. The outer branch sees only its exclusive Chebyshev shell. Final force heads start at zero, giving the exact zero-score baseline while hidden features retain their random initialization. This architecture is antisymmetric by construction, but exact recovery of every path contribution is not assumed.

In [ ]:
torch.manual_seed(TRAIN_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(TRAIN_SEED)
pair_rng = torch.Generator(device="cpu").manual_seed(TRAIN_SEED + 1)
model = MultiScaleShellForceCNEEP2D(opt).to(DEVICE)
# Begin at the zero-score baseline, avoiding large random extensive scores.
for branch in model.branches:
    torch.nn.init.zeros_(branch.force_head[-1].weight)
    if branch.force_head[-1].bias is not None:
        torch.nn.init.zeros_(branch.force_head[-1].bias)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

def pair_batch(video, replica_index, time_index):
    return torch.stack((video[replica_index, time_index],
                        video[replica_index, time_index + 1]), dim=1).to(DEVICE)

def random_pairs(video, batch_size):
    replica = torch.randint(video.shape[0], (batch_size,), generator=pair_rng)
    time = torch.randint(video.shape[1] - 1, (batch_size,), generator=pair_rng)
    return pair_batch(video, replica, time)

def ordered_pairs(video, batch_size):
    per_replica = video.shape[1] - 1
    for start in range(0, video.shape[0] * per_replica, batch_size):
        index = torch.arange(start, min(start + batch_size, video.shape[0] * per_replica))
        yield index // per_replica, index % per_replica

def alpha_neep_terms(score):
    if ALPHA == 0:
        return -score + torch.expm1(-score)
    if not -1 < ALPHA < 0:
        raise ValueError("Choose ALPHA=0 or -1 < ALPHA < 0.")
    return -torch.expm1(ALPHA * score) / ALPHA + torch.expm1(-(1 + ALPHA) * score) / (1 + ALPHA)

def branch_scores(batch):
    return model(batch) * AREA  # API spatial mean -> extensive branch increment

@torch.no_grad()
def evaluate_objective(video):
    model.eval()
    total, count = 0.0, 0
    for replica, time in ordered_pairs(video, EVAL_BATCH_SIZE):
        score = branch_scores(pair_batch(video, replica, time)).sum(dim=1)
        total += alpha_neep_terms(score).double().sum().item()
        count += len(score)
    return total / count

with torch.no_grad():
    check = pair_batch(train_video, torch.tensor([0, 1]), torch.tensor([0, 0]))
    branch = branch_scores(check)
    raw_maps = model(check, return_maps=True)
    assert branch.shape == (2, MODEL_MAX_DISTANCE + 1)
    torch.testing.assert_close(branch, raw_maps.sum(dim=(-2, -1)), atol=2e-5, rtol=2e-5)
    torch.testing.assert_close(branch_scores(check.flip(1)), -branch, atol=2e-5, rtol=2e-5)
    torch.testing.assert_close(
        raw_maps.sum(dim=(1, 2, 3)),
        branch.sum(dim=1), atol=2e-5, rtol=2e-5,
    )
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print("Pair boundaries, map normalization, and reversal checks passed.")

In [ ]:
history = {"iteration": [0], "train": [], "validation": []}
initial_train = evaluate_objective(train_video)
initial_val = evaluate_objective(val_video)
if not (math.isfinite(initial_train) and math.isfinite(initial_val)):
    raise FloatingPointError("Nonfinite initial training or validation objective.")
history["train"].append(initial_train)
history["validation"].append(initial_val)
best_val, best_iteration = initial_val, 0
checkpoint_path = RESULT_DIR / "best_model.pt"

def save_checkpoint(iteration, val_loss):
    torch.save({
        "state_dict": model.state_dict(), "model_options": vars(opt),
        "iteration": iteration, "validation_loss": val_loss,
        "physics": asdict(config), "sample_dt": SAMPLE_DT,
        "cache_ids": dict(cache_ids), "simulator_sha256": SIMULATOR_HASH,
        "simulation_backend": RESOLVED_SIMULATION_BACKEND, "cuda_source_sha256": CUDA_SOURCE_HASH,
        "train_seed": TRAIN_SEED, "alpha": ALPHA,
        "score_convention": "spatial_mean_times_area", "lattice_area": AREA,
        "initialization": "zero final force heads; random hidden features",
    }, checkpoint_path)

save_checkpoint(0, initial_val)
recent_losses = []
progress = trange(1, N_ITER + 1, desc="ensemble alpha-NEEP", disable=SMOKE)
for iteration in progress:
    model.train()
    optimizer.zero_grad(set_to_none=True)
    score = branch_scores(random_pairs(train_video, BATCH_SIZE)).sum(dim=1)
    loss = alpha_neep_terms(score).mean()
    if not torch.isfinite(loss):
        raise FloatingPointError("Nonfinite alpha-NEEP loss; lower the learning rate or sampling interval.")
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM, error_if_nonfinite=True)
    optimizer.step()
    recent_losses.append(float(loss.detach()))
    if iteration % RECORD_EVERY == 0 or iteration == N_ITER:
        val_loss = evaluate_objective(val_video)
        if not math.isfinite(val_loss):
            raise FloatingPointError("Nonfinite validation loss.")
        history["iteration"].append(iteration)
        history["train"].append(float(np.mean(recent_losses)))
        history["validation"].append(val_loss)
        recent_losses.clear()
        if val_loss < best_val:
            best_val, best_iteration = val_loss, iteration
            save_checkpoint(iteration, val_loss)
        progress.set_postfix(validation=f"{val_loss:.4g}", best=best_iteration)

# This checkpoint was created locally by this run and only contains tensors/basic metadata.
checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
model.load_state_dict(checkpoint["state_dict"])
model.eval()
print(f"Best iteration={best_iteration}, validation objective={best_val:.6g}, zero-score baseline=0")
if best_val >= -1e-6:
    warnings.warn("Validation did not improve over the zero-score baseline; interpret plots as diagnostics of unresolved learning.")
fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
ax.plot(history["iteration"], history["train"], ".-", label="training")
ax.plot(history["iteration"], history["validation"], ".-", label="independent validation")
ax.axhline(0, color="k", lw=0.8)
ax.set(xlabel="iteration", ylabel="alpha-NEEP objective")
ax.legend(); ax.grid(alpha=0.2)
fig.savefig(RESULT_DIR / "training_objective.png", dpi=150)
plt.show()

## 5. Held-out observed irreversibility and the microscopic reference

The exact reference accumulates every microscopic jump between saved frames. NEEP sees only two snapshots: hidden hops, returns, and rotations can remove observable irreversibility. Its optimal score is a two-frame log-probability ratio, including the stationary distribution boundary term. Therefore an individual predicted score is **not** a supervised medium-EP label. For stationary ensembles, the exact pair KL is bounded by full-path irreversibility on average; a finite learned estimate need not enforce this bound.

The comparison below reports full-system entropy per physical time. Error bars use independent **trajectory means**, not adjacent frames treated as independent samples. A small number of replicas or a short observation window gives imprecise uncertainty estimates.

In [ ]:
@torch.no_grad()
def predict_ensemble(video):
    model.eval()
    branch_chunks, local_chunks = [], []
    for replica, time in ordered_pairs(video, EVAL_BATCH_SIZE):
        batch = pair_batch(video, replica, time)
        raw = model(batch, return_maps=True)
        if not torch.isfinite(raw).all():
            raise FloatingPointError("Nonfinite held-out prediction.")
        branches = raw.sum(dim=(-2, -1))
        local = raw.sum(dim=1)
        torch.testing.assert_close(local.sum(dim=(-2, -1)), branches.sum(dim=1), atol=2e-5, rtol=2e-5)
        branch_chunks.append(branches.cpu().numpy())
        local_chunks.append(local.cpu().numpy())
    shape = (video.shape[0], video.shape[1] - 1)
    return (np.concatenate(branch_chunks).reshape(*shape, MODEL_MAX_DISTANCE + 1),
            np.concatenate(local_chunks).reshape(*shape, config.lattice_size, config.lattice_size))

pred_branches, pred_local = predict_ensemble(test_video)
pred_total = pred_branches.sum(axis=-1)
reference = results["test"]
true_total = reference.medium_ep
true_local = reference.medium_ep_maps
np.testing.assert_allclose(pred_local.sum(axis=(-2, -1)), pred_total, rtol=2e-5, atol=2e-5)
duration = reference.times[-1] - reference.times[0]
true_replica_rates = true_total.sum(axis=1) / duration
pred_replica_rates = pred_total.sum(axis=1) / duration
rate_summary = {
    "exact_model_medium_ep_rate_mean_sem": mean_sem(true_replica_rates),
    "learned_two_frame_score_rate_mean_sem": mean_sem(pred_replica_rates),
    "rate_difference_mean_sem": mean_sem(pred_replica_rates - true_replica_rates),
}
test_objective = evaluate_objective(test_video)
print(json.dumps(rate_summary, indent=2))
print(f"Held-out alpha-NEEP objective={test_objective:.6g}; zero-score baseline=0.")
if pred_total.mean() <= 0:
    warnings.warn("Held-out mean score is nonpositive; positive observed irreversibility was not resolved.")

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
time_end = reference.times[1:]
true_cumulative = true_total.cumsum(axis=1)
pred_cumulative = pred_total.cumsum(axis=1)
for values, label in ((true_cumulative, "exact model medium"), (pred_cumulative, "learned observed")):
    average = values.mean(axis=0)
    error = values.std(axis=0, ddof=1) / np.sqrt(TEST_M)
    axes[0].plot(time_end, average, label=label)
    axes[0].fill_between(time_end, average - error, average + error, alpha=0.2)
axes[0].set(xlabel="physical time after burn", ylabel="cumulative full-system entropy")
axes[0].legend(); axes[0].grid(alpha=0.2)
replicas = np.arange(TEST_M)
axes[1].plot(replicas, true_replica_rates, "o", label="exact model medium")
axes[1].plot(replicas, pred_replica_rates, "x", label="learned observed")
axes[1].set(xlabel="independent test replica", ylabel="entropy / physical time")
axes[1].set_xticks(replicas)
axes[1].legend(); axes[1].grid(alpha=0.2)
fig.savefig(RESULT_DIR / "held_out_entropy_rates.png", dpi=150)
plt.show()

## 6. Signed Chebyshev-shell spectrum and local maps

A shell with radius \(k\) is the boundary of a \((2k+1)\times(2k+1)\) square. The simulator interaction only samples axial forward sites inside those shells; the learned kernel can inspect the entire shell.

The exact algebraic decomposition assigns the constant forward/backward log-rate ratio to \(k=0\) and each interaction coefficient to its distance \(k\). This convention is a rate decomposition, not a unique physical localization. Learned branch contributions are jointly fitted and may redistribute across shells. The extra learned shell can capture displacement/end-point dependencies; its presence is not an additional simulator interaction.

Both spectra are **signed** and use linear axes. Negative branches are retained. A nonzero learned \(k>1\) branch alone is not a proof that this shell is necessary: compare independent runs with zero interaction, shifted interaction weights, and models retrained without selected shells.

Exact local maps split each hop's entropy equally between its two endpoints, so reversal negates the map. Learned maps use the convolution center. Their pixel values need not agree even when their sums are close.

In [ ]:
distances = np.arange(MODEL_MAX_DISTANCE + 1)
learned_replica_shell_rates = pred_branches.sum(axis=1) / duration
exact_replica_shell_rates = np.zeros_like(learned_replica_shell_rates, dtype=np.float64)
exact_width = reference.shell_ep.shape[-1]
exact_replica_shell_rates[:, :exact_width] = reference.shell_ep.sum(axis=1) / duration
learned_shell_mean = learned_replica_shell_rates.mean(axis=0)
learned_shell_sem = learned_replica_shell_rates.std(axis=0, ddof=1) / np.sqrt(TEST_M)
exact_shell_mean = exact_replica_shell_rates.mean(axis=0)
exact_shell_sem = exact_replica_shell_rates.std(axis=0, ddof=1) / np.sqrt(TEST_M)
np.testing.assert_allclose(exact_shell_mean.sum(), true_replica_rates.mean(), atol=1e-9)
np.testing.assert_allclose(learned_shell_mean.sum(), pred_replica_rates.mean(), rtol=2e-5, atol=2e-5)

fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
ax.bar(distances - 0.19, exact_shell_mean, width=0.38, yerr=exact_shell_sem,
       capsize=3, label="exact log-rate decomposition")
ax.bar(distances + 0.19, learned_shell_mean, width=0.38, yerr=learned_shell_sem,
       capsize=3, label="learned branch decomposition")
ax.axhline(0, color="k", lw=0.8)
ax.set(xlabel="exclusive Chebyshev shell k", ylabel="signed entropy / physical time",
       title="Different decompositions; bars are not shell-wise target labels")
ax.set_xticks(distances); ax.legend(); ax.grid(axis="y", alpha=0.2)
fig.savefig(RESULT_DIR / "chebyshev_shell_spectrum.png", dpi=150)
plt.show()

replica_index = 0
frame_index = (N_FRAMES - 1) // 2
local_reference = true_local[replica_index, frame_index]
local_prediction = pred_local[replica_index, frame_index]
limit = max(float(np.abs(local_reference).max()), float(np.abs(local_prediction).max()), 1e-8)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5), constrained_layout=True)
axes[0].imshow(reference.states[replica_index, frame_index] >= 0, cmap="Greys", vmin=0, vmax=1)
axes[0].set_title("Held-out occupancy")
for ax, field, label in zip(axes[1:], (local_reference, local_prediction),
                           ("Exact symmetric-endpoint map", "Learned convolution-center map")):
    im = ax.imshow(field, cmap="RdBu_r", vmin=-limit, vmax=limit)
    ax.set_title(label)
    fig.colorbar(im, ax=ax, shrink=0.8)
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
fig.savefig(RESULT_DIR / "local_entropy_maps.png", dpi=150)
plt.show()

## 7. Save reproducible diagnostics

The summary retains simulator/source hashes, seeds, split sizes, cache identifiers, learning options, and the model checkpoint. The compressed held-out arrays preserve trajectory and time axes. For a controlled interaction comparison, rerun from the configuration cell with all shell weights zero, then with only one nonzero distance; the cache keys change automatically.

In [ ]:
summary = {
    "physics": asdict(config),
    "simulation": {
        "algorithm": "exact Gillespie", "n_frames": N_FRAMES,
        "sample_dt": SAMPLE_DT, "sampling_frequency": SAMPLING_FREQUENCY,
        "observation_time": OBSERVATION_TIME, "burn_time": BURN_TIME,
        "workers": WORKERS, "split_seeds": SPLIT_SEEDS,
        "backend_request": SIMULATION_BACKEND, "backend": RESOLVED_SIMULATION_BACKEND,
        "cuda_batch_size": CUDA_BATCH_SIZE,
        "replicas": {"train": TRAIN_M, "validation": VAL_M, "test": TEST_M},
        "replica_seeds": {name: np.asarray(result.seeds).tolist() for name, result in results.items()},
    },
    "training": {
        "model": "MultiScaleShellForceCNEEP2D", "options": vars(opt),
        "iterations": N_ITER, "batch_size": BATCH_SIZE, "seed": TRAIN_SEED,
        "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
        "alpha": ALPHA, "best_iteration": best_iteration,
        "best_validation_objective": best_val, "test_objective": test_objective,
        "score_convention": "branch spatial means times lattice area; raw maps are local entropy increments",
        "lattice_area": AREA,
        "initialization": "zero final force heads; random hidden features",
    },
    "cache_ids": cache_ids, "simulator_sha256": SIMULATOR_HASH,
    "cuda_source_sha256": CUDA_SOURCE_HASH,
    "model_sha256": hashlib.sha256((ROOT / "models" / "NEEP_ShellForce_2D.py").read_bytes()).hexdigest(),
    "rates": rate_summary, "drift_diagnostics": drift_diagnostics,
    "learned_shell_rate_mean": learned_shell_mean.tolist(),
    "learned_shell_rate_sem": learned_shell_sem.tolist(),
    "exact_shell_rate_mean": exact_shell_mean.tolist(),
    "exact_shell_rate_sem": exact_shell_sem.tolist(),
    "smoke": SMOKE,
    "interpretation": "Unsupervised two-frame observed irreversibility compared with exact microscopic model medium EP; distinct shell/local decompositions.",
}
(RESULT_DIR / "summary.json").write_text(json.dumps(summary, indent=2, allow_nan=False), encoding="utf-8")
np.savez_compressed(
    RESULT_DIR / "held_out_diagnostics.npz",
    times=reference.times, replica_seeds=reference.seeds,
    true_total_increment=true_total, learned_total_increment=pred_total,
    true_local_increment=true_local, learned_local_increment=pred_local,
    true_shell_increment=reference.shell_ep, learned_shell_increment=pred_branches,
    true_replica_rates=true_replica_rates, learned_replica_rates=pred_replica_rates,
    history_iteration=np.asarray(history["iteration"]),
    history_train=np.asarray(history["train"]),
    history_validation=np.asarray(history["validation"]),
)
plt.close("all")
print("Saved checkpoint and diagnostics:", RESULT_DIR)
print("Smoke complete." if SMOKE else "Pilot run complete; inspect validation and relaxation before increasing the run size.")